In [1]:
# %%
import os, pickle, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold
from scipy.stats import wilcoxon
from catboost import CatBoostRegressor, Pool

warnings.filterwarnings('ignore')

# ---- пути ----
BASE_PATH = "/Users/egorilin/Desktop/COMA"
BASE = Path(BASE_PATH)
ANALYSIS_DIR = BASE / 'analysis'
ANALYSIS_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Output: {ANALYSIS_DIR}')

# ---- константы домена ----
TARGET_COLUMNS = [
    'S. aureus ATCC 43300 Activity MIC, мг/л',
    'S. aureus ATCC 43300 Activity MBC, мг/л',
    'E. coli ATCC 25922 Activity MIC, мг/л',
    'E. coli ATCC 25922 Activity MBC, мг/л',
]
SHORT_NAMES = ['SA_MIC', 'SA_MBC', 'EC_MIC', 'EC_MBC']
ORIG_COL_MAP = {sn: TARGET_COLUMNS[i] for i, sn in enumerate(SHORT_NAMES)}
OOD_COL_MAP = {
    'SA_MIC': 'MIC Sa',
    'SA_MBC': 'MBC Sa',
    'EC_MIC': 'MIC Ec',
    'EC_MBC': 'MBC Ec',
}

Device: cpu
Output: /Users/egorilin/Desktop/COMA/analysis


In [2]:
# %%
class AdvancedContrastiveEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dims=[512, 384, 256],
                 projection_dim=128, dropout=0.3):
        super().__init__()
        layers = []; prev = input_dim
        for hd in hidden_dims:
            layers.extend([nn.Linear(prev, hd), nn.BatchNorm1d(hd),
                           nn.ReLU(), nn.Dropout(dropout)])
            prev = hd
        self.encoder = nn.Sequential(*layers)
        self.projection_head = nn.Sequential(
            nn.Linear(hidden_dims[-1], projection_dim), nn.BatchNorm1d(projection_dim),
            nn.ReLU(), nn.Dropout(dropout * 0.5),
            nn.Linear(projection_dim, projection_dim), nn.BatchNorm1d(projection_dim),
        )

    def forward(self, x, return_projection=True):
        f = self.encoder(x)
        if return_projection:
            return f, self.projection_head(f)
        return f


class HardNegContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.05, base_temperature=0.07, alpha=0.3):
        super().__init__()
        self.t, self.bt, self.alpha = temperature, base_temperature, alpha
    def forward(self, features, labels):
        dev = features.device; bs = features.shape[0]
        f = F.normalize(features, dim=1)
        sim = f @ f.T / self.t
        labels = labels.view(-1, 1)
        mask_pos = torch.eq(labels, labels.T).float().to(dev)
        mask_neg = 1 - mask_pos
        logits_mask = torch.ones_like(mask_pos); logits_mask.fill_diagonal_(0)
        mask_pos *= logits_mask; mask_neg *= logits_mask
        neg = torch.exp(sim) * mask_neg
        hard_w = F.softmax(neg / self.t, dim=1)
        exp_l = torch.exp(sim) * logits_mask
        log_p = sim - torch.log(exp_l.sum(1, keepdim=True))
        mlp = (mask_pos * log_p).sum(1) / mask_pos.sum(1).clamp(min=1.0)
        pen = (hard_w * sim * mask_neg).sum(1)
        return (-(self.t / self.bt) * mlp + self.alpha * pen).mean()


class EmbDS(Dataset):
    def __init__(self, X, y, augment=True, noise=0.02):
        self.X = torch.FloatTensor(X); self.y = torch.LongTensor(y)
        self.aug, self.noise = augment, noise
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        x = self.X[i].clone()
        if self.aug and torch.rand(1).item() > 0.5:
            x = x + torch.randn_like(x) * self.noise
        return x, self.y[i]

print('✓ Архитектура и loss определены')

✓ Архитектура и loss определены


In [6]:
# %%
# In-domain
data_orig   = pd.read_excel(f'{BASE_PATH}/Test_molformer_original_biocides.xlsx').drop(['Unnamed: 0'], axis=1)
target_orig = pd.read_excel(f'{BASE_PATH}/Data_biocides.xlsx')

# OOD
ood_raw       = pd.read_excel(f'{BASE_PATH}/QACs_SMILES_MIC_MBC.xlsx')
ood_molformer = pd.read_excel(f'{BASE_PATH}/Test_molformer_ood.xlsx')
for c in ['Unnamed: 0']:
    if c in ood_molformer.columns: ood_molformer = ood_molformer.drop(columns=[c])
ood_molformer = ood_molformer.select_dtypes(include=[np.number])
ood_raw['MBC Ec'] = pd.to_numeric(ood_raw['MBC Ec'], errors='coerce')

# Scaler + cls-trained encoder
with open(f'{BASE_PATH}/scaler_classification.pkl', 'rb') as f:
    scaler_clf = pickle.load(f)

encoder_cls = AdvancedContrastiveEncoder(input_dim=768, hidden_dims=[512, 384, 256],
                                         projection_dim=128, dropout=0.3).to(device)
encoder_cls.load_state_dict(torch.load(f'{BASE_PATH}/encoder_borderline_smote.pth',
                                       map_location=device))
encoder_cls.eval()

# Эмбеддинги обоих сплитов через cls-encoder
def extract_emb(enc, X_scaled):
    enc.eval()
    with torch.no_grad():
        return enc(torch.FloatTensor(X_scaled).to(device), return_projection=False).cpu().numpy()

orig_scaled = scaler_clf.transform(data_orig)
ood_scaled  = scaler_clf.transform(ood_molformer)
orig_emb_cls = extract_emb(encoder_cls, orig_scaled)
ood_emb_cls  = extract_emb(encoder_cls, ood_scaled)

molformer_orig = data_orig.values.astype(float)
molformer_ood  = ood_molformer.values.astype(float)

print(f'data_orig:    {data_orig.shape}')
print(f'ood_raw:      {ood_raw.shape}')
print(f'orig_emb_cls: {orig_emb_cls.shape}')
print(f'ood_emb_cls:  {ood_emb_cls.shape}')

KeyboardInterrupt: 

In [ ]:
# %%
def build_pairs(mol_feat, target_df, col_map):
    rows = []
    for ti, sn in enumerate(SHORT_NAMES):
        col = col_map[sn]
        if col not in target_df.columns: continue
        vals = pd.to_numeric(target_df[col], errors='coerce')
        idx = np.where(vals.notna().values)[0]
        for gi in idx:
            y = float(vals.values[gi])
            if y <= 0: continue
            rows.append({
                'task': ti, 'sn': sn, 'gi': int(gi),
                'y_log2': float(np.log2(y)), 'y_raw': y,
                'mol_feat': mol_feat[gi],
            })
    return rows

def to_arrays(pairs):
    return {
        'mol':    np.stack([r['mol_feat'] for r in pairs]),
        'task':   np.array([r['task']     for r in pairs]),
        'gi':     np.array([r['gi']       for r in pairs]),
        'y_log2': np.array([r['y_log2']   for r in pairs]),
        'y_raw':  np.array([r['y_raw']    for r in pairs]),
    }

# Пары с MolFormer и emb_cls (одинаковый порядок строк, разные фичи)
A_in_mf  = to_arrays(build_pairs(molformer_orig, target_orig, ORIG_COL_MAP))
A_in_em  = to_arrays(build_pairs(orig_emb_cls,   target_orig, ORIG_COL_MAP))
A_ood_mf = to_arrays(build_pairs(molformer_ood,  ood_raw,    OOD_COL_MAP))
A_ood_em = to_arrays(build_pairs(ood_emb_cls,    ood_raw,    OOD_COL_MAP))

D_MF  = A_in_mf['mol'].shape[1]   # 768
D_EMB = A_in_em['mol'].shape[1]   # 256

# Combined фичи [molformer | emb] — для SHAP
X_in_combined  = np.hstack([A_in_mf['mol'],  A_in_em['mol']])
X_ood_combined = np.hstack([A_ood_mf['mol'], A_ood_em['mol']])

mf_cols  = [f'mf_{i}'  for i in range(D_MF)]
emb_cols = [f'emb_{i}' for i in range(D_EMB)]
ALL_COLS = mf_cols + emb_cols

print(f'in-domain pairs: {len(A_in_mf["task"])}')
print(f'OOD pairs:       {len(A_ood_mf["task"])}')
print(f'D_MF={D_MF}, D_EMB={D_EMB}, combined={X_in_combined.shape[1]}')